## **Music Recommendation Algorithm Project**
</br>
Info here

***Model***

Details

---
### **1. Imports**

In [1]:
# Importing sys to ensure proper environment setup
import sys

print(sys.version_info)

sys.version_info(major=3, minor=11, micro=14, releaselevel='final', serial=0)


In [2]:
# Importing pandas and numpy for numerical analysis
# Importing pyplot and seaborn to visualize the data
# Importing os, pathlib, and warnings for functionality, faster loading, flagging exceptions, etc.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker, pylab
from matplotlib.legend import Legend
import statistics
from scipy.stats import skew, kurtosis, trim_mean
import seaborn as sns
import lightgbm as lgb
import os # possibly remove
from pathlib import Path, PureWindowsPath
from scipy.stats import multivariate_normal, norm, trim_mean, zscore
import warnings
warnings.filterwarnings('ignore')

# Importing sklearn items for preprocessing, model training & testing
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error, classification_report,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# Different label assignment (assign_labels="cluster_qr") as deterministic partitioning alternative
from sklearn.cluster import KMeans, AffinityPropagation, DBSCAN, SpectralClustering
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.decomposition import PCA

# Makes graphs appear in line
%matplotlib inline

sns.set(style="whitegrid", palette="Set3", font_scale=1.25)    # alt muted, spectral, tab20c

print("Setup Complete")

Setup Complete


---
### **2. Load Data & Quick Review**

In [3]:
# Explicitly noting path as being in Windows format to avoid issues with backsplash
#filename = PureWindowsPath("..Data\tcc_ceds_music.csv")

# Convert path to the correct format
#file_path = Path(filename)
rec_test = open("C:\\Users\\winni\\music-rec-algo\\Data\\rec_test_data.csv")
clean_b = open("C:\\Users\\winni\\music-rec-algo\\Data\\clean_b.csv")

# Loading data as a DataFrame
df_rec_test = pd.read_csv(rec_test) 
df_clean_b = pd.read_csv(clean_b)

# Using head() function to display the first five rows of the data
print("Heads")
print("Recommendation Test Data:", df_rec_test.head())
print("Cleaned Data:", df_clean_b.head())

Heads
Recommendation Test Data:    Unnamed: 0       artist_name           track_name  release_date   genre  \
0       76885          godsmack               immune          1998    rock   
1       65394      dennis brown        second chance          1993  reggae   
2       10980  the black crowes          sister luck          1990     pop   
3         842   jerry lee lewis  your cheating heart          1960     pop   
4        2764         paul anka             eso beso          1966     pop   

                                              lyrics  len    dating  violence  \
0  come world society futher place home land deat...   74  0.000907  0.348191   
1  maybe maybe treat good feel second best girl s...   43  0.001224  0.029943   
2  worry sick eye hurt rest head life outside gir...   54  0.001120  0.482490   
3  cheat heart weep sleep sleep come night cheat ...   25  0.204740  0.002506   
4  beso kiss beso kiss know samba bossanova close...   97  0.001170  0.001170   

   world/lif

In [4]:
# Using shape() function to return a tuple listing number of rows and columns in the data

print("Shape of Rec Test:", df_rec_test.shape)
print("Shape of Clean Data:", df_clean_b.shape)

Shape of Rec Test: (10, 25)
Shape of Clean Data: (28372, 40)


In [5]:
# Using info() function to view column names, data types, and other relevant information

df_rec_test.info()

df_clean_b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                10 non-null     int64  
 1   artist_name               10 non-null     object 
 2   track_name                10 non-null     object 
 3   release_date              10 non-null     int64  
 4   genre                     10 non-null     object 
 5   lyrics                    10 non-null     object 
 6   len                       10 non-null     int64  
 7   dating                    10 non-null     float64
 8   violence                  10 non-null     float64
 9   world/life                10 non-null     float64
 10  night/time                10 non-null     float64
 11  shake the audience        10 non-null     float64
 12  family/gospel             10 non-null     float64
 13  romantic                  10 non-null     float64
 14  communication

In [6]:
# Updating column names to remove whitespace and erroroneous characters on rec_test

df_rec_test.columns = df_rec_test.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(':', '')
df_rec_test.columns = df_rec_test.columns.str.replace(')', '').str.replace('-', '_').str.replace('/', '_')
print("Updated Column Names:", df_rec_test.columns)
print(df_rec_test.info())

Updated Column Names: Index(['Unnamed_0', 'artist_name', 'track_name', 'release_date', 'genre',
       'lyrics', 'len', 'dating', 'violence', 'world_life', 'night_time',
       'shake_the_audience', 'family_gospel', 'romantic', 'communication',
       'obscene', 'music', 'movement_places', 'light_visual_perceptions',
       'family_spiritual', 'like_girls', 'sadness', 'feelings', 'topic',
       'age'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed_0                 10 non-null     int64  
 1   artist_name               10 non-null     object 
 2   track_name                10 non-null     object 
 3   release_date              10 non-null     int64  
 4   genre                     10 non-null     object 
 5   lyrics                    10 non-null     object 
 6   len                     

In [7]:
# Placeholder



--- 
### **3. Feature Engineering: Train/Test/Split**

In [8]:
# Creating copies of datasets for multiple model testings

X_train = df_clean_b.copy()
X_rec = df_rec_test.copy()

##### **<p style="text-align:center;">3A. Principal Component Analysis (PCA)</p>**    
Performing encoding separately to minimize chance of error.

In [9]:
# Principal Component Analysis (PCA)

pca = PCA()
pca.fit(X_train)

plt.figure(figsize=(14,6))
plt.title("Principal Component Analysis")
plt.plot(pca.explained_variance_ration_)
plt.legend("Explained Variance")
plt.xlabel("N Components")
plt.ylabel("Explained Variance Ratio")
plt.show()

ValueError: could not convert string to float: 'mukesh'

In [ ]:
print("PCA X-train:", pca.fit(X_train))

# Creating optimal n_components
X = df_clean_b.copy()

pca = PCA(n_components=4)
X_centered = X - X.mean(axis=0)
pca.fit(X_centered)
print("PCA X-fit center:", pca.fit(X_centered))

X_pca = pca.transform(X_centered)
print("PCA X-fit center transform:", X_pca)

In [ ]:
# Plotting explained variance ratio for the components
explained_variance_ratio = pca.explained_variance_ratio_

plt.figure(figsize=(10,6))
plt.bar(range(1, len(explained_variance_ratio) + 1, explained_variance_ratio, alpha=0.8, align="center"))
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Centered PCA")
plt.xticks(range(1, len(explained_variance_ratio) + 1))
plt.show()

##### **<p style="text-align:center;">3B. The Body Beautiful: Elbow, Silhouette & More</p>**    
Performing primliminary clustering via the Elbow Method and related methologies.

In [ ]:
# Inertia score will be stored in this list
inertia_score = []

# Defining cluster values to test
# KMeans model testing 2-12 clusters
k_values = range(2, 13)

# Iterating for loop of cluster possibilities
for k in k_values:
    # Creating initial KMeans model with max # of clusters
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=12)

    # Fitting model on training feature matrix
    kmeans.fit(X_train)

    # Beginning intertia score (how compact clusters are)
    inertia_score.append(kmeans.inertia_)

plt.figure(figsize=(14,6))

plt.plot(range(2, 13), inertia_score, linewidth=2, marker=8)
plt.title("Elbow Plot [KMeans] Inertia Score")
plt.xlabel("K")
plt.ylabel("Inertia Score")
plt.xticks(X_train)
plt.show()

# saving plot as image in docs folder
# plt.savefig("C:\\Users\\winni\\music-rec-algo\\FOLDER")

In [ ]:
# Silhouette score will be stored in this list
silhouette = []

# Iterating for loop of cluster possibilities
for k in range(2, 13):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=12)

    predictions = kmeans.fit_predict(X.values)
    silhouette.append(metrics.silhouette_score(X, predictions))

plt.figure(figsize=(14, 6))

plt.plot(range(2, 13), silhouette, linewidth=2, marker=8)
plt.title("Silhouette Plot [KMeans]")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.xticks(k_values)
plt.show()

# saving plot as image in docs folder
# plt.savefig("C:\\Users\\winni\\music-rec-algo\\FOLDER")

In [ ]:
# Creating 2D cluster visual Elbow/Silhouette to help finalize decision

labels = kmeans.fit_predict(X.values)
print(labels)

kmeans.cluster_centers_

plt.figure(figsize=(15, 9))

plt.scatter(X.value[:, 0], X.values[:, 1], c=kmeans.labels_, s=106)
plt.scatter(kmeans.cluster_centers_{:, 0], kmeans.cluster_centers_[:, 1], color='red', s=225)
plt.title("Cluster of Songs")
plt.xlabel("Score")
plt.ylabel("Len")
plt.show()

In [ ]:
# Creating 2D cluster visual of PCA to help finalize decision
kmeans_pca = KMeans(n_clusters=2, random_state=42)
kmeans_pca.fit(X_pca)

labels_pca = kmeans.fit_predict(X_pca)
print(labels_pca)

kmeans_pca.cluster_centers_

plt.figure(figsize=(15, 9))

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_pca.labels_, s=106)
plt.scatter(kmeans_pca.cluster_centers_{:, 0], kmeans_pca.cluster_centers_[:, 1], color='red', s=225)
plt.title("Cluster of Songs (PCA)")
plt.xlabel("Score")
plt.ylabel("Len")
plt.show()

##### *Elbow & Silhouette Insights*

abc

##### **<p style="text-align:center;">3C. Choosing Clusters</p>**    
Performing encoding separately to minimize chance of error.

In [ ]:
# Label encoding the genre column to convert categorical data into numerical data for analysis

le = LabelEncoder()
df1['genre_enc'] = le.fit_transform(df1['genre'])

print("Label Encoder - Category Mapping:")
print(dict(enumerate(le.classes_)))

# One-Hot Encoding via get_dummies to drop 'genre' column)
df1 = pd.get_dummies(df1, columns=['genre'], drop_first=True, dtype=int)

print("DF after get_dummies:/n", df1.head(5))

In [ ]:
# Dropping genre_enc column as it is no longer needed after encoding

df1 = df1.drop(columns=['genre_enc'])
print("DF after dropping genre_enc:/n", df1.head(5))

In [ ]:
# Strong positive correlations found between topical columns () & their related topics

# df1_encorr = df1.drop(['topic_music', ])

---
### **4. Modeling Music Features**

In [ ]:
# Model test ... apply KMeans to music

# result should be songs

In [ ]:
# Grouping rating columns together to represent music track attributes
rating = ['dating', 'family_gospel', 'communication', 'family_spiritual', 'like_girls', 
          'shake_the_audience', 'movement_places', 'light_visual_perceptions']

X = songs['rating']

# Apply KMeans clustering with k=3 on selected features
kmeans_music = kmeans(n_clusters=3, random_state=42)
kmeans_music.fit(X)

# Box plot of rating columns
plt.figure(figsize=(12, 6))
sns.boxplot(data=df['rating'], orient='h', palette='spectral')
plt.title('Box Plot of Rating Columns')
plt.xlabel('Rating Score')
plt.ylabel('Columns')
plt.show()

In [ ]:
# Using sample of 10 unlabeled songs to see number of cluster generation
sample_songs = songs.sample(10)

selected_predictors = sample_songs[rating]
sample_labels = kmeans_music.predict(selected_predictors)

# sample_songs[pred] = sample_labels

In [ ]:
# Handling outliers with IQR capping at 1.5*IQR boundary (Winsorization)
# Done in lieu of dropping rows, thereby preserving data

for col in skewed_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df1_log_t[col] = df1_log_t[col].clip(lower=lower, upper=upper)

# Using log1p again to handle zero values safely
# Only appling it to positively skewed columns (skew > 0.5)
for col in skewed_cols:
    if skewness[col] > 0.5:
        df1_log_t[col] = np.log1p(df1_log_t[col])
    elif skewness[col] < -0.5:
        # Reflect then log for negatively skewed
        df1_log_t[col] = np.log1p(df1_log_t[col].max() - df1_log_t[col])

In [ ]:
# Checking for any missing values after skew & transformations
print(df1_log_t.isnull().sum())

# For numeric columns, fill with median (robust to outliers)
num_cols = df1_log_t.select_dtypes(include='number').columns
df1_log_t[num_cols] = df1_log_t[num_cols].fillna(df1_log_t[num_cols].median())

# For categorical columns, fill with mode
cat_cols = df1_log_t.select_dtypes(include='object').columns
for col in cat_cols:
    df1_log_t[col] = df1_log_t[col].fillna(df1[col].mode()[0])

In [ ]:
# Scaling columns
cols_to_scale = df1_log_t.select_dtypes(include=['int', 'float']).columns.tolist()
scaler = StandardScaler()

df1_scaled = df1_log_t.copy()
df1_scaled[cols_to_scale] = scaler.fit_transform(df1_log_t[cols_to_scale])

print(df1_scaled.head(5))

---
### **5. Verifying Changes**

In [ ]:
# Confirm no missing values remain
print("Missing values after cleaning:")
print(df1_scaled.isnull().sum())
print()

# Confirm shape is intact
print("DataFrame shape:\n", df1_scaled.shape)

# Numeric columns check
print("\nNumeric column stats:")
print(df1_scaled[num_cols].describe())

---
### **6. Save Cleaned CSV**

In [ ]:
# Explicity noting path as being in Windows format so I can use forward slash
clean = PureWindowsPath("C:\\Users\\Winni\\music-rec-algo\\Data\\cleaned.csv")

# Convert path to the correct format
file_path = Path(clean)

# Saving data as a DataFrame
df1_scaled.to_csv(file_path, index=False)
# index=False prevents pandas from saving row numbers as a column
print("Cleaned data saved successfully!")

In [ ]:
# Most reliable confirmation is to reload saved file and review it
clean = PureWindowsPath("C:\\Users\\Winni\\music-rec-algo\\Data\\cleaned.csv")
file_path = Path(clean)

df1_clean = pd.read_csv(file_path)

print("Shape:", df1_clean.shape)
print("Missing values:\n", df1_clean.isnull().sum())
print("\nFinal columns:", df1_clean.columns.tolist())
print("\nPreview:\n", df1_clean.head(5))